## 01- Transformer Classifier Baseline
In this notebook, we will do a baseline using a mix of classicial and neural method (Transformer-based sentence embeddings and logistic regression classifier)

In [1]:
# !pip install sentence-transformers

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics import (
    classification_report,
    mean_squared_error,
)
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.model_selection import cross_val_score

/Users/la/miniconda3/envs/mds/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
train_fname = "../data/splits/train_combined.csv"
dev_fname = "../data/splits/dev_combined.csv"

train_df = pd.read_csv(train_fname)
dev_df = pd.read_csv(dev_fname)

print("Train df=", len(train_df))
print("Dev df =", len(dev_df))

Train df= 348
Dev df = 88


In [10]:
train_df.head()

,id,text,mmse,target
0,172-1,oh ‡ the water is running out_of the sink .\nm...,24.0,AD
1,212-3,mhm .\nthe stool the boy's on the stool reach...,19.0,AD
2,506-0,well ‡ the boy is in the cookie jar &=laughs ....,25.0,AD
3,NaN,Start whenever you want cookie jar a land stan...,30.0,CN
4,704-0,well ‡ the little kid's falling off his chair ...,23.0,AD


In [11]:
dev_df.head()

,id,text,mmse,target
0,221-1,alright .\naction .\noh ‡ well ‡ the little gi...,30.0,AD
1,650-0,uh a boy is &-uh puttin(g) somethin(g) in the ...,24.0,AD
2,291-1,(.) &-uh the boy in the cookie jar .\nuh the l...,20.0,AD
3,488-0,there she's washing the dishes .\nand there th...,20.0,AD
4,145-1,hm .\ntouching lip .\nraising arm .\nis that w...,30.0,CN


### Pre-processing
We will lowercase the text for better efficiency

In [13]:
train_df["text_lower"] = train_df["text"].apply(lambda text: text.lower().strip())
dev_df["text_lower"] = dev_df["text"].apply(lambda text: text.lower().strip())

### Apply Sentence Transformer

We can use Sentence Transformer to convert text to a fixed-length embedding vector

In [14]:
# vectorizer = SentenceTransformer("sentence-t5-large")  # works best
vectorizer = SentenceTransformer("intfloat/e5-mistral-7b-instruct")  # works best

Loading checkpoint shards: 100%|██████████| 2/2 [00:32<00:00, 16.27s/it]


In [15]:
X_train = vectorizer.encode(train_df["text_lower"])
X_dev = vectorizer.encode(dev_df["text_lower"])

In [16]:
print(X_train.shape)

(348, 4096)


In [17]:
print(X_dev.shape)

(88, 4096)


### Train and Evaluate MLP Classifier (AD vs CN)

We can now train a multi layer perceptron classifier using processed X_train

In [18]:
def train_eval_model(model, X_train, y_train, X_dev, y_dev, cv, is_classifier=True):
    print("Training neural model...")
    model.fit(X_train, y_train)
    print("Completed")

    y_pred = model.predict(X_train)
    if is_classifier:
        print(classification_report(y_train, y_pred))
    else:
        train_rmse = np.sqrt(mean_squared_error(y_train, y_pred))
        print("Train RMSE: ", train_rmse)

    print("Performing cross-validation...")
    scores = cross_val_score(model, X_train, y_train, cv=cv)
    print("Average =", np.mean(scores))

    y_pred = model.predict(X_dev)
    if is_classifier:
        print(classification_report(y_dev, y_pred))
    else:
        dev_rmse = np.sqrt(mean_squared_error(y_dev, y_pred))
        print("Dev RMSE: ", dev_rmse)

In [19]:
solver = "lbfgs"
alpha = 1e-5
learning_rate_init = 0.0001
hidden_size = 500
cv = 10
max_iter = 500
random_state = 42

In [20]:
y_train = train_df["target"]
y_dev = dev_df["target"]
clf = MLPClassifier(
    solver=solver,
    alpha=alpha,
    hidden_layer_sizes=(hidden_size, hidden_size, hidden_size),
    learning_rate_init=learning_rate_init,
    max_iter=max_iter,
    random_state=random_state,
)

train_eval_model(clf, X_train, y_train, X_dev, y_dev, cv)

Training neural model...
Completed
              precision    recall  f1-score   support

          AD       1.00      1.00      1.00       199
          CN       1.00      1.00      1.00       149

    accuracy                           1.00       348
   macro avg       1.00      1.00      1.00       348
weighted avg       1.00      1.00      1.00       348

Performing cross-validation...
Average = 0.74109243697479
              precision    recall  f1-score   support

          AD       0.84      0.82      0.83        50
          CN       0.77      0.79      0.78        38

    accuracy                           0.81        88
   macro avg       0.80      0.80      0.80        88
weighted avg       0.81      0.81      0.81        88



### Train and Evaluate MLP Regressor (MMSE scores)

We now continue to train a multi layer perceptron regressor using processed X_train

In [21]:
y_train = train_df["mmse"]
y_dev = dev_df["mmse"]
clf = MLPRegressor(
    solver=solver,
    alpha=alpha,
    hidden_layer_sizes=(hidden_size, hidden_size, hidden_size),
    learning_rate_init=learning_rate_init,
    max_iter=max_iter,
    random_state=random_state,
)
train_eval_model(clf, X_train, y_train, X_dev, y_dev, cv, is_classifier=False)

Training neural model...


/Users/la/miniconda3/envs/mds/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


Completed
Train RMSE:  0.358143970805131
Performing cross-validation...


/Users/la/miniconda3/envs/mds/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/Users/la/miniconda3/envs/mds/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/Users/la/miniconda3/envs/mds/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs faile

Average = 0.3173108137569308
Dev RMSE:  5.0360361942669
